# Step 1: Preprocess Image-Caption Pairs

Extract features from (image, caption) pairs:
- Caption -> PhoBERT embeddings [max_len, 768]
- Image -> ResNet50 features [2048]

- Input: `./pairs/pairs_{train,dev,test}.json`
- Output: `./processed/coolant_{train,dev,test}.h5`

In [1]:
import sys
import os
import gc
import numpy as np
import torch
import h5py
from pathlib import Path
from tqdm import tqdm

project_root = Path().absolute().parent.parent
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / "src"))

print(f"Project root: {project_root}")

Project root: g:\My Drive\Thesis_Final\fake-new-detection


In [2]:
# Config
CONFIG = {
    "text_model": "vinai/phobert-base",
    "image_model": "resnet50",
    "language": "vi",
    "max_length": 128,  # Captions are short (not full articles)
    "batch_size": 32,
}

PAIRS_DIR = Path("./pairs")
OUTPUT_DIR = Path("./processed")
OUTPUT_DIR.mkdir(exist_ok=True)

device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Device: {device}")

Device: cuda


In [3]:
# Initialize preprocessor
from src.preprocessing import CombinedPreprocessor

preprocessor = CombinedPreprocessor(
    text_model_name=CONFIG["text_model"],
    image_model_name=CONFIG["image_model"],
    language=CONFIG["language"],
    max_text_length=CONFIG["max_length"],
    device=device,
)
print("Preprocessor ready")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: vinai/phobert-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
lm_head.layer_norm.bias         | UNEXPECTED |  | 
lm_head.decoder.bias            | UNEXPECTED |  | 
lm_head.layer_norm.weight       | UNEXPECTED |  | 
lm_head.dense.bias              | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
lm_head.decoder.weight          | UNEXPECTED |  | 
lm_head.dense.weight            | UNEXPECTED |  | 
lm_head.bias                    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
c:\Users\tungh\.conda\envs\fake_news\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\tungh\.conda\envs\fake_news\Lib\site-packages\torchvision\models\_utils.py:

Preprocessor ready


In [4]:
# Load pairs and preprocess each split
from src.processing.coolant import PairExtractor

BATCH_SIZE = CONFIG["batch_size"]

for split in ["train", "dev", "test"]:
    pairs_path = PAIRS_DIR / f"pairs_{split}.json"
    if not pairs_path.exists():
        print(f"{split}: pairs file not found, skipping")
        continue

    pairs = PairExtractor.load_pairs(str(pairs_path))
    captions = [p["caption"] for p in pairs]
    image_paths = [p["image_path"] for p in pairs]
    article_ids = [p["article_idx"] for p in pairs]
    total = len(pairs)
    n_batches = (total + BATCH_SIZE - 1) // BATCH_SIZE

    print(f"\nProcessing {split}: {total} pairs, {n_batches} batches")

    all_caption_feat, all_image_feat = [], []

    for i in tqdm(range(n_batches), desc=split):
        s, e = i * BATCH_SIZE, min((i + 1) * BATCH_SIZE, total)

        cap_feat = preprocessor.text_preprocessor.extract_token_embeddings(captions[s:e])
        img_feat = preprocessor.image_preprocessor.extract_features(image_paths[s:e])

        all_caption_feat.append(cap_feat)
        all_image_feat.append(img_feat)

        del cap_feat, img_feat
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    caption_features = np.vstack(all_caption_feat)
    image_features = np.vstack(all_image_feat)
    article_id_array = np.array(article_ids)

    # Save HDF5
    hdf5_path = OUTPUT_DIR / f"coolant_{split}.h5"
    chunk_size = min(64, total)

    with h5py.File(hdf5_path, "w") as f:
        f.create_dataset(
            "caption_features", data=caption_features,
            chunks=(chunk_size, caption_features.shape[1], caption_features.shape[2]),
            compression="gzip", compression_opts=4,
        )
        f.create_dataset(
            "image_features", data=image_features,
            chunks=(chunk_size, image_features.shape[1]),
            compression="gzip", compression_opts=4,
        )
        f.create_dataset("article_ids", data=article_id_array)
        f.attrs["n_samples"] = total
        f.attrs["caption_shape"] = caption_features.shape[1:]
        f.attrs["image_shape"] = image_features.shape[1:]

    size_mb = hdf5_path.stat().st_size / (1024 * 1024)
    print(f"  caption={caption_features.shape}, image={image_features.shape}")
    print(f"  Saved: {hdf5_path} ({size_mb:.1f} MB)")

    del all_caption_feat, all_image_feat, caption_features, image_features
    gc.collect()

print("\nAll splits preprocessed.")


Processing train: 3537 pairs, 111 batches


train: 100%|██████████| 111/111 [02:32<00:00,  1.37s/it]


  caption=(3537, 128, 768), image=(3537, 2048)
  Saved: processed\coolant_train.h5 (261.0 MB)

Processing dev: 1054 pairs, 33 batches


dev: 100%|██████████| 33/33 [00:39<00:00,  1.20s/it]


  caption=(1054, 128, 768), image=(1054, 2048)
  Saved: processed\coolant_dev.h5 (73.4 MB)

Processing test: 876 pairs, 28 batches


test: 100%|██████████| 28/28 [00:31<00:00,  1.14s/it]


  caption=(876, 128, 768), image=(876, 2048)
  Saved: processed\coolant_test.h5 (64.5 MB)

All splits preprocessed.


In [5]:
# Verify HDF5 files
for split in ["train", "dev", "test"]:
    path = OUTPUT_DIR / f"coolant_{split}.h5"
    if path.exists():
        with h5py.File(path, "r") as f:
            print(f"{split}: caption={f['caption_features'].shape}, image={f['image_features'].shape}, articles={len(set(f['article_ids'][:]))}")

print("\nReady for Step 2 (training).")

train: caption=(3537, 128, 768), image=(3537, 2048), articles=1653
dev: caption=(1054, 128, 768), image=(1054, 2048), articles=519
test: caption=(876, 128, 768), image=(876, 2048), articles=412

Ready for Step 2 (training).
